In [1]:
import subprocess
import librosa
import numpy as np
import json
from pytube import YouTube
import os

def download_audio(youtube_url, output_path):
    yt = YouTube(youtube_url)
    audio_stream = yt.streams.filter(only_audio=True).first()
    downloaded_file = audio_stream.download(filename=output_path)
    return downloaded_file

def convert_to_ogg(input_path, output_path):
    # Convert to OGG using ffmpeg
    subprocess.run(['ffmpeg', '-i', input_path, '-c:a', 'libvorbis', output_path], check=True)
    return output_path

def load_and_preprocess_audio(audio_path, target_sr=22050):
    # Load audio file at a reduced sample rate
    y, sr = librosa.load(audio_path, sr=target_sr)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    log_S = librosa.power_to_db(S, ref=np.max)
    return log_S, sr


def dynamic_time_warping(S1, S2):
    _, wp = librosa.sequence.dtw(X=S1, Y=S2, metric='euclidean')
    return wp

def adjust_timestamps(wp, timestamps, sr, offset):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    for entry in timestamps:
        original_frame = int((entry['t'] + offset) * sr / 512)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * 512 / sr - offset
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    return adjusted_timestamps


# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=JDvckfO1_GQ'
youtube_url2 = 'https://www.youtube.com/watch?v=JVBzE0mUlSs'

# Download audio files
audio_path1 = download_audio(youtube_url1, 'youtube_audio1.mp3')
audio_path2 = download_audio(youtube_url2, 'youtube_audio2.mp3')

# Convert to OGG
ogg_path1 = convert_to_ogg(audio_path1, 'youtube_audio1.ogg')
ogg_path2 = convert_to_ogg(audio_path2, 'youtube_audio2.ogg')

# Load and preprocess both audio recordings in OGG format
S1, sr1 = load_and_preprocess_audio(ogg_path1)
S2, sr2 = load_and_preprocess_audio(ogg_path2)

# Dynamic Time Warping
warping_path = dynamic_time_warping(S1, S2)


# Sample JSON timestamps for the first recording
timestamps_json = '''
[{"t":0,"mix":0},{"t":1.498,"mix":1},{"t":9.313,"mix":2},{"t":17.32,"mix":3},{"t":24.421,
"mix":4},{"t":32.978,"mix":5},{"t":40.798,"mix":6},{"t":49.285,"mix":7},{"t":56.787,"mix":8},
{"t":65.319,"mix":9},{"t":72.935,"mix":10},{"t":81.899,"mix":11},{"t":88.848,"mix":12},{"t":97.245,
"mix":13},{"t":105.682,"mix":14},{"t":114.185,"mix":15},{"t":121.288,"mix":16},{"t":130.686,
"mix":17},{"t":137.922,"mix":18},{"t":146.628,"mix":19},{"t":153.583,"mix":20},{"t":162.361,
"mix":21},{"t":170.995,"mix":22},{"t":179.733,"mix":23},{"t":186.793,"mix":24},{"t":196.292,
"mix":25},{"t":204.738,"mix":26},{"t":213.692,"mix":27},{"t":222.042,"mix":28},{"t":230.335,
"mix":29},{"t":240.107,"mix":30},{"t":248.105,"mix":31},{"t":255.864,"mix":32},{"t":267.932,
"mix":33},{"t":278.696,"mix":34},{"t":286.466,"mix":35},{"t":305.898,"mix":36}]
'''  # Use the full JSON
timestamps = json.loads(timestamps_json)

# Sample JSON timestamps for the first recording (assumed already loaded)
offset = 0.93
adjusted_timestamps = adjust_timestamps(warping_path, timestamps, sr1, offset)

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']
new_adjusted_timestamps = [{"t": item["t"] + initial_offset, "mix": item["mix"]} for item in adjusted_timestamps]

# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps, indent=4))
print(f"Offset: {initial_offset}")

# Cleanup downloaded and converted files
os.remove(audio_path1)
os.remove(audio_path2)
os.remove(ogg_path1)
os.remove(ogg_path2)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

[
    {
        "t": 0.0,
        "mix": 0
    },
    {
        "t": 0.8591383219954648,
        "mix": 1
    },
    {
        "t": 10.309659863945578,
        "mix": 2
    },
    {
        "t": 17.205986394557826,
        "mix": 3
    },
    {
        "t": 23.916553287981863,
        "mix": 4
    },
    {
        "t": 31.718458049886625,
        "mix": 5
    },
    {
        "t": 38.22004535147392,
        "mix": 6
    },
    {
        "t": 45.11637188208616,
        "mix": 7
    },
    {
        "t": 52.43065759637188,
        "mix": 8
    },
    {
        "t": 60.34866213151927,
        "mix": 9
    },
    {
        "t": 66.98956916099772,
        "mix": 10
    },
    {
        "t": 75.0701133786848,
        "mix": 11
    },
    {
        "t": 82.26829931972789,
        "mix": 12
    },
    {
        "t": 90.99900226757369,
        "mix": 13
    },
    {
        "t": 97.89532879818593,
        "mix": 14
    },
    {
        "t": 104.67555555555555,
        "mix": 15
    },
    {
   